<div dir="rtl">

# 🌈 03 - Chroma Vector Store in LangChain

## ما هو Chroma DB؟
- **Chroma** هو مستودع متجهات مفتوح المصدر مصمم خصيصاً لتطبيقات الـ AI والـ LLMs مع التركيز على تجربة المطور وسهولة الاستخدام.
- يأتي بتكامل رسمي مع LangChain عبر حزمة `langchain-chroma`.

---

### 🌟 أهم مميزات Chroma:
1. **تخزين دائم تلقائي (Built-in Persistence)**: يدعم حفظ البيانات على القرص تلقائياً مع محرك SQLite ومحرك متجهات داخلي.
2. **فلترة ميتاداتا قوية ومتطورة**: دعم المعاملات المنطقية (`$and`, `$or`, `$eq`, `$gte`, `$in`).
3. **عمليات التعديل والحذف (CRUD)**: سهولة تحديث المستندات، حذفها بناءً على معرّفات (IDs)، والاستعلام عن عدد العناصر.
4. **دعم المجموعات (Collections)**: عزل المستندات والمشاريع في مجموعات منفصلة داخل نفس قاعدة البيانات.

</div>


<div dir="rtl">

### 1️⃣ تجهيز المكتبات ونموذج التضمين

</div>


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

load_dotenv(find_dotenv())

# تهيئة نموذج التضمين
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("✅ تم تحميل نموذج التضمين بنجاح!")


<div dir="rtl">

### 2️⃣ إنشاء مجموعة دائمة على القرص (Persistent Collection)
نقوم بتحديد مسار مجلد الحفظ `persist_directory` واسم المجموعة `collection_name`.

</div>


In [ ]:
persist_dir = "../../data/chroma_db"
os.makedirs(persist_dir, exist_ok=True)

# تهيئة قاعدة بيانات Chroma دائمة
chroma_db = Chroma(
    collection_name="tech_articles",
    embedding_function=embeddings,
    persist_directory=persist_dir
)

print(f"✅ تم تهيئة Chroma بنجاح في المسار: {os.path.abspath(persist_dir)}")


<div dir="rtl">

### 3️⃣ إضافة مستندات مع معرّفات مخصصة (Custom IDs) وميتاداتا غنية

</div>


In [ ]:
docs = [
    Document(
        page_content="نماذج لاما 3 و DeepSeek تمثل قفزة نوعية في النماذج مفتوحة المصدر وكفاءة الاستدلال.",
        metadata={"category": "LLM", "author": "Ali", "year": 2024, "rating": 5}
    ),
    Document(
        page_content="تقنيات LangGraph تسمح ببناء وكلاء أذكياء بنظام الحالات والحلقات التكرارية (Stateful Multi-Agent).",
        metadata={"category": "Agents", "author": "Sara", "year": 2024, "rating": 4}
    ),
    Document(
        page_content="محرك Docker يسهل عزل التطبيقات وتشغيلها في بيئات حاويات معزولة ومستقرة.",
        metadata={"category": "DevOps", "author": "Ali", "year": 2022, "rating": 5}
    ),
    Document(
        page_content="تتيح Kubernetes إدارة ونشر مجموعات الحاويات تلقائياً وبكفاءة وتوسع هائل.",
        metadata={"category": "DevOps", "author": "Omar", "year": 2023, "rating": 4}
    )
]

doc_ids = ["doc_llm_1", "doc_agent_1", "doc_devops_1", "doc_devops_2"]

# إضافة المستندات مع المعرفات المحددة
chroma_db.add_documents(documents=docs, ids=doc_ids)

print(f"📊 إجمالي المستندات في المجموعة: {chroma_db._collection.count()}")


<div dir="rtl">

### 4️⃣ الفلترة المتقدمة بواسطة الميتاداتا (Advanced Metadata Filtering)
يدعم Chroma شروط الفلترة المتقدمة عبر معاملات مثل `$eq`, `$and`, `$gte`.

</div>


In [ ]:
# 1. فلترة بسيطة: البحث عن فئة DevOps فقط
devops_results = chroma_db.similarity_search(
    "كيف نقوم بنشر الحاويات وإدارتها؟",
    k=2,
    filter={"category": "DevOps"}
)

print("🎯 نتائج فئة DevOps فقط:")
for doc in devops_results:
    print(f"• {doc.page_content} (Author: {doc.metadata['author']})")

print("\n" + "="*50 + "\n")

# 2. فلترة مركبة: الكاتب Ali مع سنة 2024
complex_filter = {
    "$and": [
        {"author": {"$eq": "Ali"}},
        {"year": {"$gte": 2024}}
    ]
}

filtered_results = chroma_db.similarity_search(
    "ما هي أحدث التطورات في النماذج؟",
    k=2,
    filter=complex_filter
)

print("🎯 نتائج الفلترة المركبة (Ali + 2024+):")
for doc in filtered_results:
    print(f"• {doc.page_content} ({doc.metadata})")


<div dir="rtl">

### 5️⃣ تحديث وحذف المستندات بالـ ID (Update & Delete)

</div>


In [ ]:
# حذف مستند باستخدام معرّفه
chroma_db.delete(ids=["doc_devops_1"])
print(f"🗑️ تم حذف المستند. العدد المتبقي: {chroma_db._collection.count()}")

# التحقق من أن المستند لم يعد يظهر في البحث
check_results = chroma_db.similarity_search("Docker الحاويات", k=2)
for doc in check_results:
    print(f"• {doc.page_content}")
